In [1]:
from sklearn.decomposition import PCA
from indicators.volume import MFIIndicator, OBVIndicator, VWAPIndicator
from indicators.volatilidade import BollingerBandsIndicator, DonchianChannelIndicator
from indicators.types import IndicatorConfig, IndicatorType
from indicators.tendencia import IchimokuCloudIndicator
from indicators.niveis import FibonacciRetracementIndicator
from indicators.momento import CCIIndicator, ROCIndicator, RSIIndicator
from indicators.medias_moveis import ADXIndicator, EMAIndicator, MACDIndicator, SMAIndicator
from events.types import Direction
from events.fill_event import FillEvent
from data_loader.loader import DataLoader
import datetime
import random
import gc
from typing import Optional, Tuple
import gymnasium as gym
import numpy as np
import optuna
import polars as pl
from gymnasium import spaces
from gymnasium.wrappers import TimeLimit
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import pathlib
import yaml
import torch

from env.trading_env import CustomTradingEnv

Módulo 'indicators' não encontrado ou incompleto (Erro: cannot import name 'momentum' from 'indicators' (/workspaces/VIII-Desafio-de-Ciencia-de-Dados/indicators/__init__.py)). Funcionalidade `add_technical_indicators` estará desabilitada.
2025-05-13 12:50:28.909941: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-13 12:50:28.929205: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747140628.944571   56905 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747140628.949626   56905 cuda_blas.cc:1407] Unable to register cuBLAS

# Pré-processamento

In [2]:
dl = DataLoader()
df = dl.load_from_parquet(
    filepath="/workspaces/BackTesting/data/AAPL_1minute.parquet")
dl.preprocess(df)
# dl.add_technical_indicators(df=df) # Removido para evitar duplicidade

df.head()

date,open,high,low,close,volume
datetime[μs],f64,f64,f64,f64,f64
2020-05-11 08:00:00,77.785,77.785,77.785,77.785,1496.0
2020-05-11 08:06:00,77.8375,77.9125,77.8375,77.9125,2224.0
2020-05-11 08:08:00,77.8375,77.8425,77.785,77.7875,16836.0
2020-05-11 08:12:00,77.785,77.785,77.785,77.785,4000.0
2020-05-11 08:13:00,77.7775,77.7775,77.7775,77.7775,760.0


Indicadores

In [3]:
lista_indicadores = []

lista_indicadores.append(
    MACDIndicator(IndicatorConfig(type=IndicatorType.MACD, params=[12, 26, 9]))
)

lista_indicadores.append(
    RSIIndicator(IndicatorConfig(type=IndicatorType.RSI, params=[14]))
)

lista_indicadores.append(
    BollingerBandsIndicator(IndicatorConfig(
        type=IndicatorType.BB, params=[20, 2]))
)

lista_indicadores.append(
    SMAIndicator(IndicatorConfig(type=IndicatorType.SMA, params=[20]))
)

lista_indicadores.append(
    IchimokuCloudIndicator(IndicatorConfig(type=IndicatorType.ICHIMOKU,params=[12, 58, 96]))
)

df_base = df.clone()
for ind in lista_indicadores:
    df = df.join(ind.calculate(df_base), on="date", how="left")

df.head(30)

date,open,high,low,close,volume,MACD_12_26_9,MACDSignal_12_26_9,MACDHist_12_26_9,rsi_14,BB_Middle_20,BB_Upper_20,BB_Lower_20,sma_20,TENKAN_12,KIJUN_58,SENKOU_A_12_58,SENKOU_B_96,CHIKOU_58
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2020-05-11 08:00:00,77.785,77.785,77.785,77.785,1496.0,null,null,null,17.418012,null,null,null,null,null,null,null,null,77.175
2020-05-11 08:06:00,77.8375,77.9125,77.8375,77.9125,2224.0,null,null,null,17.418012,null,null,null,null,null,null,null,null,77.1325
2020-05-11 08:08:00,77.8375,77.8425,77.785,77.7875,16836.0,null,null,null,17.418012,null,null,null,null,null,null,null,null,77.1725
2020-05-11 08:12:00,77.785,77.785,77.785,77.785,4000.0,null,null,null,17.418012,null,null,null,null,null,null,null,null,77.185
2020-05-11 08:13:00,77.7775,77.7775,77.7775,77.7775,760.0,null,null,null,17.418012,null,null,null,null,null,null,null,null,77.2275
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2020-05-11 09:28:00,77.2875,77.2875,77.2875,77.2875,652.0,-0.123068,null,null,21.210448,77.435,77.779261,77.090739,77.435,77.28375,null,null,null,77.175
2020-05-11 09:37:00,77.3325,77.3325,77.3325,77.3325,640.0,-0.114444,null,null,31.073083,77.414375,77.728548,77.100202,77.414375,77.2625,null,null,null,77.175
2020-05-11 09:42:00,77.375,77.375,77.375,77.375,6976.0,-0.102994,null,null,38.85754,77.399375,77.688829,77.109921,77.399375,77.2625,null,null,null,77.1625


Preencher os NaNs

In [4]:
df = df.fill_null(strategy="forward")

df = df.fill_null(0.0)

Padronização

In [5]:
df = df.with_columns([
    pl.col("open").log().alias("open_log"),
    pl.col("high").log().alias("high_log"),
    pl.col("low").log().alias("low_log"),
    pl.col("close").log().alias("close_log"),
])

df

date,open,high,low,close,volume,MACD_12_26_9,MACDSignal_12_26_9,MACDHist_12_26_9,rsi_14,BB_Middle_20,BB_Upper_20,BB_Lower_20,sma_20,TENKAN_12,KIJUN_58,SENKOU_A_12_58,SENKOU_B_96,CHIKOU_58,open_log,high_log,low_log,close_log
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2020-05-11 08:00:00,77.785,77.785,77.785,77.785,1496.0,0.0,0.0,0.0,17.418012,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,77.175,4.353949,4.353949,4.353949,4.353949
2020-05-11 08:06:00,77.8375,77.9125,77.8375,77.9125,2224.0,0.0,0.0,0.0,17.418012,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,77.1325,4.354623,4.355586,4.354623,4.355586
2020-05-11 08:08:00,77.8375,77.8425,77.785,77.7875,16836.0,0.0,0.0,0.0,17.418012,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,77.1725,4.354623,4.354688,4.353949,4.353981
2020-05-11 08:12:00,77.785,77.785,77.785,77.785,4000.0,0.0,0.0,0.0,17.418012,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,77.185,4.353949,4.353949,4.353949,4.353949
2020-05-11 08:13:00,77.7775,77.7775,77.7775,77.7775,760.0,0.0,0.0,0.0,17.418012,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,77.2275,4.353852,4.353852,4.353852,4.353852
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-01-01 00:37:00,250.4101,250.42,250.4101,250.42,822.0,-0.009658,-0.007147,-0.002511,46.659979,250.439825,250.487256,250.392394,250.439825,250.45005,250.525,250.3575,250.29,250.41,5.5231,5.52314,5.5231,5.52314
2025-01-01 00:41:00,250.42,250.42,250.42,250.42,524.0,-0.009776,-0.007673,-0.002103,46.659979,250.43893,250.487183,250.390677,250.43893,250.45005,250.525,250.3575,250.29,250.41,5.52314,5.52314,5.52314,5.52314
2025-01-01 00:48:00,250.45,250.45,250.45,250.45,107.0,-0.007364,-0.007611,0.000247,51.722888,250.43978,250.488192,250.391368,250.43978,250.45005,250.525,250.3575,250.29,250.41,5.523259,5.523259,5.523259,5.523259


Divisão e Padronização dos Dados

1. Identificar colunas a serem escaladas
Todas as colunas, exceto 'date' e as OHLC originais.
Isso irá incluir: 'volume', todos os indicadores técnicos e as com sufixo '_log'

In [6]:
cols_to_scale = [
    col for col in df.columns if col not in ["date", "open", "high", "low", "close"]
]
print(f"Colunas identificadas para escalonamento: {cols_to_scale}")

Colunas identificadas para escalonamento: ['volume', 'MACD_12_26_9', 'MACDSignal_12_26_9', 'MACDHist_12_26_9', 'rsi_14', 'BB_Middle_20', 'BB_Upper_20', 'BB_Lower_20', 'sma_20', 'TENKAN_12', 'KIJUN_58', 'SENKOU_A_12_58', 'SENKOU_B_96', 'CHIKOU_58', 'open_log', 'high_log', 'low_log', 'close_log']


Começar setando algumas configurações, puxar do 'config.yaml'

In [7]:
CFG = yaml.safe_load(open("config.yaml"))
SEED = CFG["seed"]
def set_seed(s):
    np.random.seed(s); random.seed(s); torch.manual_seed(s)

set_seed(SEED)
pathlib.Path("models").mkdir(exist_ok=True)

In [8]:
TRAIN_END    = (CFG["train_end"])
TEST_START   = (CFG["test_start"])
TEST_END     = (CFG["test_end"])
VAL_START    = TRAIN_END
VAL_END      = TEST_START

In [9]:
df_train = df.filter(pl.col("date") <= pl.lit(TRAIN_END))
means, stds = df_train.select([
    pl.mean(col).alias(col) for col in df_train.columns if col not in ["date"]
]).to_dict(), df_train.select([
    pl.std(col).alias(col) for col in df_train.columns if col not in ["date"]
]).to_dict()

In [10]:
df_train = df.filter(pl.col("date") <= pl.lit(TRAIN_END))

df_validation = df.filter(
    (pl.col("date") >= pl.lit(VAL_START))
    & (pl.col("date") <= pl.lit(VAL_END))
)

df_test = df.filter(
    (pl.col("date") >= pl.lit(TEST_START))
    & (pl.col("date") <= pl.lit(TEST_END))
)

3. Padronização dos dados

Função para aplicar a padronização (StandardScaler) usando médias/stds do treino

In [11]:
def apply_scaling_polars(dataframe, means_dict, stds_dict, columns_list):
    scaling_expressions = []
    for col_name in columns_list:
        if (
            col_name in dataframe.columns
            and col_name in means_dict
            and col_name in stds_dict
        ):
            mean = means_dict[col_name]
            std = stds_dict[col_name]

            if mean is None or std is None:
                # Coluna no treino era toda nula ou teve problema no cálculo de mean/std
                # print(f"  Aviso: Média/Std nula para {col_name} no treino. Coluna será preenchida com 0.0 no dataset {dataframe.shape[0]} linhas.")
                scaling_expressions.append(pl.lit(0.0).alias(col_name))
            elif (
                std > 1e-8
            ):  # Desvio padrão utilizável para evitar divisão por zero ou valores muito pequenos
                scaling_expressions.append(
                    ((pl.col(col_name) - mean) / std).alias(col_name)
                )
            # Desvio padrão muito pequeno ou zero (coluna constante no treino)
            elif mean is not None:  # Se std é ~0 mas média existe, apenas centraliza
                # print(f"  Aviso: Std muito pequeno para {col_name} ({std}). Coluna será apenas centralizada (valor - média_treino).")
                scaling_expressions.append(
                    (pl.col(col_name) - mean).alias(col_name))
            else:  # Média e Std são None ou Std é ~0 e Média é None - preenche com 0
                scaling_expressions.append(pl.lit(0.0).alias(col_name))
        # else: # Opcional: aviso se uma coluna esperada não estiver no dataframe atual
        # print(f"  Aviso: Coluna {col_name} não encontrada no dataframe atual para aplicar escala.")

    if scaling_expressions:
        return dataframe.with_columns(scaling_expressions)
    return dataframe

Função para aplicar Min-Max Scaling (-1 a 1) usando min/max do treino

In [12]:
def apply_minmax_scaling_polars(df, mins, maxs, cols):
    exprs = []
    for c in cols:
        if c in df.columns and c in mins and c in maxs:  # Adicionada checagem se c está em mins e maxs
            lo, hi = mins[c], maxs[c]
            if lo is not None and hi is not None:  # Checar se min/max foram calculados
                if hi - lo > 1e-12:                   # coluna não-constante
                    # Garantir float na multiplicação/divisão
                    exprs.append(((pl.col(c)-lo)/(hi-lo)*2.0-1.0).alias(c))
                else:                                 # constante → sempre 0
                    exprs.append(pl.lit(0.0).alias(c))
            else:
                # Caso min/max não foram calculados para a coluna (ex: não estava no treino)
                exprs.append(pl.lit(0.0).alias(c))
        # else: # Opcional: Aviso se a coluna não estiver no dataframe ou nos dicionários
            # print(f"Aviso: Coluna {c} não encontrada ou sem min/max para MinMaxScaling.")

    if exprs:
        return df.with_columns(exprs)
    return df


In [13]:
train_mins = {}
train_maxs = {}

print("\nCalculando Mínimos e Máximos do conjunto de TREINO para Min-Max Scaling...")
for col_name in cols_to_scale:
    if col_name in df_train.columns:
        min_val = df_train.select(pl.col(col_name).min()).item()
        max_val = df_train.select(pl.col(col_name).max()).item()
        train_mins[col_name] = min_val
        train_maxs[col_name] = max_val
    else:
        print(
            f"  Aviso: Coluna {col_name} não encontrada em df_train para calcular min/max."
        )


Calculando Mínimos e Máximos do conjunto de TREINO para Min-Max Scaling...


In [14]:
print("\nAplicando padronização aos conjuntos de dados...")
# Aplicar a padronização (substituindo as colunas originais pelas padronizadas)
# Polars é copy-on-write, então df_train, etc. serão novos DataFrames após with_columns
df_train = apply_minmax_scaling_polars(
    df_train, train_mins, train_maxs, cols_to_scale)
df_validation = apply_minmax_scaling_polars(
    df_validation, train_mins, train_maxs, cols_to_scale
)
df_test = apply_minmax_scaling_polars(
    df_test, train_mins, train_maxs, cols_to_scale)

print(
    "\nVerificação dos DataFrames Padronizados (primeiras 3 linhas de algumas colunas):"
)
# Mostra 'date' e até 4 colunas escaladas
cols_to_show = ["date"] + \
    [col for col in cols_to_scale if col in df_train.columns][:4]

if len(df_train) > 0:
    print("\nDataFrame de Treino Padronizado:")
    print(df_train.select(cols_to_show).head(3))
if len(df_validation) > 0:
    print("\nDataFrame de Validação Padronizado:")
    print(df_validation.select(cols_to_show).head(3))
if len(df_test) > 0:
    print("\nDataFrame de Teste Padronizado:")
    print(df_test.select(cols_to_show).head(3))


Aplicando padronização aos conjuntos de dados...

Verificação dos DataFrames Padronizados (primeiras 3 linhas de algumas colunas):

DataFrame de Treino Padronizado:
shape: (3, 5)
┌─────────────────────┬───────────┬──────────────┬────────────────────┬──────────────────┐
│ date                ┆ volume    ┆ MACD_12_26_9 ┆ MACDSignal_12_26_9 ┆ MACDHist_12_26_9 │
│ ---                 ┆ ---       ┆ ---          ┆ ---                ┆ ---              │
│ datetime[μs]        ┆ f64       ┆ f64          ┆ f64                ┆ f64              │
╞═════════════════════╪═══════════╪══════════════╪════════════════════╪══════════════════╡
│ 2020-05-11 08:00:00 ┆ -0.999845 ┆ 0.089876     ┆ 0.074524           ┆ 0.139205         │
│ 2020-05-11 08:06:00 ┆ -0.999764 ┆ 0.089876     ┆ 0.074524           ┆ 0.139205         │
│ 2020-05-11 08:08:00 ┆ -0.998139 ┆ 0.089876     ┆ 0.074524           ┆ 0.139205         │
└─────────────────────┴───────────┴──────────────┴────────────────────┴──────────────────┘



Fill NaNs

In [15]:
# df já contém OHLC + volume + indicadores + colunas _log
FEATURES = [
    c for c in df.columns
    if c not in ("date",            # não é timestamp
                 "open", "high",
                 "low",  "close")   # não são os preços crus
]


# Treinamento

In [16]:
def make_env(df_split):
    def _thunk():
        env = CustomTradingEnv(df_split, features=FEATURES, window_size=30)
        env = TimeLimit(env, max_episode_steps=6_000)
        env = Monitor(env)   # <-- aqui
        return env
    env = DummyVecEnv([_thunk])
    env = VecNormalize(env,
                       norm_obs=True,
                       norm_reward=True,
                       clip_obs=10.)
    return env

In [17]:
def objective(trial: optuna.Trial) -> float:
    """Função objetivo para otimização de hiperparâmetros da PPO."""

    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    n_steps = trial.suggest_categorical("n_steps", [256, 512, 1024, 2048, 4096])
    gamma = trial.suggest_float("gamma", 0.9, 0.9999, log=True)
    ent_coef = trial.suggest_float("ent_coef", 0.00000001, 0.1, log=True)
    clip_range = trial.suggest_categorical("clip_range", [0.1, 0.2, 0.3])
    gae_lambda = trial.suggest_float("gae_lambda", 0.8, 0.99)
    vf_coef = trial.suggest_float("vf_coef", 0.1, 1.0)

    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256, 512])
    if batch_size > n_steps:
        batch_size = n_steps

    train_env = make_env(df_train)
    eval_env = make_env(df_validation)

    model = PPO(
        policy="MlpPolicy",
        env = train_env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        gamma=gamma,
        ent_coef=ent_coef,
        clip_range=clip_range,
        gae_lambda=gae_lambda,
        vf_coef=vf_coef,
        seed = SEED,
        device = "auto",
        verbose=0,
        tensorboard_log="tb/"
    )

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=None,
        log_path=None,
        eval_freq=CFG["checkpoint_every"]//5,
        deterministic=True,
        render=False,
        n_eval_episodes=5,
    )

    TIMESTEPS_PER_TRIAL = CFG["total_timesteps"]//10
    if(TIMESTEPS_PER_TRIAL < n_steps*2):
        TIMESTEPS_PER_TRIAL = n_steps*2

    try:
        model.learn(total_timesteps=TIMESTEPS_PER_TRIAL, callback=[eval_callback])
        mean_reward = eval_callback.last_mean_reward

    except Exception as e:
        print(f"Erro ao treinar o modelo: {e}")
        mean_reward = -float("inf")

    finally:
        train_env.close()
        eval_env.close()

    return mean_reward


In [18]:
class SaveVecNormalizeCallback(BaseCallback):
    def __init__(self, save_path: str, verbose: int = 0):
        super().__init__(verbose)
        self.save_path = save_path

    def _on_step(self) -> bool:
        # Salva só quando o EvalCallback marcou um novo best
        if self.locals.get("eval_env") is not None and self.locals.get("is_success", False):
            self.training_env.save(self.save_path)
        return True

In [ ]:
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))

N_TRIALS=500
study.optimize(objective, n_trials=N_TRIALS, n_jobs=4)

the_best = study.best_trial
print("Valor(Recomenpensa Média): ", the_best.value)
print("Parâmetros do melhor trial: ")
for key, value in the_best.params.items():
    print(f"  {key}: {value}")

print("-"*100)

best_params = study.best_trial.params
# Certifique-se de que o batch_size seja válido para o n_steps encontrado
if "batch_size" in best_params and "n_steps" in best_params:
    if best_params["batch_size"] > best_params["n_steps"]:
        best_params["batch_size"] = best_params["n_steps"]


[I 2025-05-13 12:50:31,953] A new study created in memory with name: no-name-0389236d-02d7-45d7-95af-9d05161f85e9


Eval num_timesteps=20000, episode_reward=-0.00 +/- 0.00
Episode length: 6000.00 +/- 0.00
New best mean reward!
Eval num_timesteps=20000, episode_reward=0.05 +/- 0.00
Episode length: 6000.00 +/- 0.00
New best mean reward!
Eval num_timesteps=20000, episode_reward=-0.00 +/- 0.00
Episode length: 6000.00 +/- 0.00
New best mean reward!
Eval num_timesteps=20000, episode_reward=0.00 +/- 0.00
Episode length: 6000.00 +/- 0.00
New best mean reward!
